# Ablation — GCN + features par articulation

**Objectif :** Mesurer l'impact des vecteurs osseux par nœud sur le GCN.

Chaque nœud du graphe passe de 3 dims (x,y,z) à 9 dims : xyz + vecteur osseux
entrant + vecteur osseux sortant. Même topologie, mêmes hyperparamètres.

**Protocole expérimental (commun à toutes les expériences) :**
- Dataset : 7 664 images du dataset W7CH (TSL), landmarks MediaPipe (21 points × 2 mains).
- Split stratifié 80/10/10 (seed 42) sur les échantillons ORIGINAUX, **avant**
  toute augmentation → aucune fuite de données entre train et validation/test.
- Augmentation (rotation ±20°, échelle 0.85-1.15, translation ±0.1, bruit 0.02)
  appliquée **uniquement** sur le train, équilibré à 200 échantillons/classe.
- Entraînement : Adam (lr 1e-3, wd 1e-4), CrossEntropy + label smoothing 0.1,
  ReduceLROnPlateau, early stopping sur val_acc, seeds fixes.
- 56 classes effectives sur 57 (les images `metro` ne sont pas détectées par
  MediaPipe), tête de classification à 57 sorties (comme le modèle actuel).


In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
ROOT = NB_DIR
while not (ROOT / "common").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for p in [str(ROOT), str(ROOT / "common")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

from common.paths import RESULTS_DIR, MODELS_DIR, PLOTS_DIR, REPO_ROOT
from common.data import load_dataset, normalized_inputs
from common.models import (
    count_params, set_seed, BaselineMLP, LandmarkTransformer, HandGCN, HandGAT,
)
from common.train import (
    Trainer, predict, full_metrics, save_metrics,
    plot_training_curves, plot_confusion, run_torch_benchmark,
)
from common import export as ex
from common.robustness import robustness_summary

set_seed(42)

def load_class_names():
    with open(REPO_ROOT / "data" / "processed" / "class_names.txt", encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]

class_names = load_class_names()
print("torch", torch.__version__)


torch 2.13.0+cpu


## 1. Chargement des données

Le dataset préparé (split + augmentation, cf. README) est chargé. Le modèle
reçoit graphe de 21 nœuds × 2 mains, features par nœud : 9 dims.


In [2]:
dataset = load_dataset()
from common.data import joint_features
jt_train = joint_features(dataset["train"]["img_landmarks"])
jt_val   = joint_features(dataset["val"]["img_landmarks"])
jt_test  = joint_features(dataset["test"]["img_landmarks"])
X_train, y_train = torch.from_numpy(jt_train).float(), torch.from_numpy(dataset["train"]["labels"])
X_val, y_val     = torch.from_numpy(jt_val).float(),   torch.from_numpy(dataset["val"]["labels"])
X_test, y_test   = torch.from_numpy(jt_test).float(),  torch.from_numpy(dataset["test"]["labels"])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False)
print("train:", tuple(X_train.shape), "| val:", tuple(X_val.shape), "| test:", tuple(X_test.shape))


train: (13282, 2, 21, 9) | val: (767, 2, 21, 9) | test: (767, 2, 21, 9)


## 2. Architecture

GCN identique au notebook 03, entrées à 9 dimensions par nœud.

Nombre de paramètres : mesuré ci-dessous.


In [3]:
model = HandGCN(num_classes=57, hidden=64, num_layers=2, input_dim=9)
print(f"params: {count_params(model):,}")


params: 45,433


## 3. Entraînement

Mêmes hyperparamètres que le protocole commun (Adam lr 1e-3, wd 1e-4,
label smoothing 0.1, early stopping patience 20, max 100 epochs).


In [4]:
trainer = Trainer(model, patience=20)
history = trainer.fit(train_loader, val_loader, max_epochs=100, early_stop=True)
trainer.restore_best()

plot_training_curves(history, PLOTS_DIR / "training_curves" / "gcn_geo_curves.png",
                     title="Ablation — GCN + features par articulation")
print("best val acc:", trainer.best_val_acc)


  ep 1/100 train 3.0788/0.3110 val 2.3772/0.5593 (1.0s) best 0.5593


  ep 2/100 train 2.1656/0.5652 val 1.8875/0.6975 (1.5s) best 0.6975


  ep 3/100 train 1.8421/0.6534 val 1.6780/0.7471 (1.6s) best 0.7471


  ep 4/100 train 1.6831/0.7044 val 1.5116/0.8005 (1.1s) best 0.8005


  ep 5/100 train 1.5751/0.7417 val 1.4250/0.8279 (1.5s) best 0.8279


...

  ep 93/100 train 0.9887/0.9618 val 0.9902/0.9674 (1.6s) best 0.9713


  ep 94/100 train 0.9936/0.9630 val 0.9896/0.9661 (1.6s) best 0.9713


  ep 95/100 train 0.9918/0.9630 val 0.9855/0.9674 (1.0s) best 0.9713


  ep 96/100 train 0.9939/0.9605 val 0.9803/0.9674 (1.1s) best 0.9713
  early stop @ 96 (best val acc 0.9713)


best val acc: 0.9713168187744459


## 4. Évaluation sur le jeu de test

Métriques complètes : accuracy, precision/recall/F1 (macro et pondérés),
matrice de confusion, performance par classe.


In [5]:
y_pred, y_true = predict(model, test_loader)
metrics = full_metrics(y_true, y_pred, num_classes=57)
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"macro F1: {metrics['macro_f1']:.4f} | weighted F1: {metrics['f1']:.4f}")
print(f"macro precision: {metrics['macro_precision']:.4f} | macro recall: {metrics['macro_recall']:.4f}")

cm = np.array(metrics["confusion_matrix"])
plot_confusion(cm, class_names, PLOTS_DIR / "confusion_matrices" / "gcn_geo_cm.png",
               title="Ablation — GCN + features par articulation — confusion matrix (test)")
np.save(RESULTS_DIR / "confusion_gcn_geo.npy", cm)

import pandas as pd
per = pd.DataFrame({
    "class": class_names,
    "precision": metrics["per_class_precision"],
    "recall": metrics["per_class_recall"],
    "f1": metrics["per_class_f1"],
    "support": metrics["per_class_support"],
})
per.to_csv(RESULTS_DIR / "per_class_gcn_geo.csv", index=False)
print(per.sort_values("f1").head(8))


test accuracy: 0.9687
macro F1: 0.9680 | weighted F1: 0.9690
macro precision: 0.9711 | macro recall: 0.9680


     class  precision    recall        f1  support
54   metro   0.000000  0.000000  0.000000        0
17  siye7a   0.777778  0.777778  0.777778        9
39   jadda   0.789474  0.937500  0.857143       16
8    labes   1.000000  0.750000  0.857143        4
53  louage   0.750000  1.000000  0.857143        6
13     non   0.857143  0.923077  0.888889       13
46   erb3a   0.900000  0.900000  0.900000       10
5       cv   1.000000  0.818182  0.900000       11


## 5. Robustesse

Le modèle est évalué sous perturbations des landmarks appliquées **avant**
normalisation (dans l'espace image) : bruit gaussien, rotation, translation,
changement d'échelle — 5 essais par perturbation, moyenne ± écart-type.


In [6]:
def forward_fn(model, raw, batch_size=512):
    # Perturbed raw landmarks -> per-joint features (x,y,z + bone vectors)
    model.eval()
    from common.data import joint_features
    jf = joint_features(raw)
    outs = []
    with torch.no_grad():
        for i in range(0, len(jf), batch_size):
            x = torch.from_numpy(jf[i:i + batch_size]).float()
            outs.append(model(x).argmax(1).numpy())
    return np.concatenate(outs)
rob = robustness_summary(model, dataset["test"]["img_landmarks"],
                         dataset["test"]["labels"], base_acc=metrics["accuracy"],
                         forward_fn=forward_fn)


  noise_0.01: acc=0.8897 (drop=+0.0790)
  noise_0.02: acc=0.7872 (drop=+0.1815)


  noise_0.05: acc=0.4829 (drop=+0.4858)
  rotation_5: acc=0.9700 (drop=-0.0013)
  rotation_10: acc=0.9439 (drop=+0.0248)
  rotation_20: acc=0.8422 (drop=+0.1265)


  translation_0.05: acc=0.9416 (drop=+0.0271)
  translation_0.10: acc=0.9085 (drop=+0.0602)
  scale_0.90: acc=0.9622 (drop=+0.0065)
  scale_1.10: acc=0.9739 (drop=-0.0052)


## 6. Mesures mobiles

- Nombre de paramètres et taille du checkpoint PyTorch.
- Export ONNX (FP32), vérification, latence ONNX Runtime CPU (batch=1).
- Versions FP16 et INT8 (quantification dynamique) : taille, latence,
  dégradation d'accuracy sur le jeu de test.


In [7]:
sample = X_test[:1].float()
lat_torch = run_torch_benchmark(model, sample)
print(f"torch latency: {lat_torch:.3f} ms/sample")

onnx_path = ex.export_onnx(model, sample, "gcn_geo")
fp16 = ex.quantize_fp16(onnx_path, "gcn_geo")
int8 = ex.quantize_int8(onnx_path, "gcn_geo")

quantized = {}
for tag, p in [("fp32", onnx_path), ("fp16", fp16), ("int8", int8)]:
    preds = ex.ort_predict(p, X_test.numpy())
    quantized[tag] = {
        "size_mb": ex.file_size_mb(p),
        "latency_ms": ex.ort_latency(p, sample.numpy()),
        "test_acc": float((preds == y_test.numpy()).mean()),
    }
    print(f"  {tag}: size={quantized[tag]['size_mb']:.2f} MB "
          f"latency={quantized[tag]['latency_ms']:.3f} ms "
          f"acc={quantized[tag]['test_acc']:.4f}")

extra = {
    "robustness": rob,
    "epochs": trainer.epochs_run,
    "params": count_params(model),
    "best_val_acc": trainer.best_val_acc,
    "torch_latency_ms": lat_torch,
    "quantized": quantized,
}
save_metrics("gcn_geo", metrics, extra=extra)

torch_path = ex.save_torch(model, "gcn_geo")
print("saved:", torch_path, f"({ex.file_size_mb(torch_path):.2f} MB)")


torch latency: 0.267 ms/sample


  fp32: size=0.18 MB latency=0.064 ms acc=0.9687
  fp16: size=0.10 MB latency=0.065 ms acc=0.9687
  int8: size=0.06 MB latency=0.061 ms acc=0.9687
metrics saved -> D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\results\metrics_gcn_geo.json
saved: D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\models\gcn_geo.pth (0.18 MB)


## 7. Conclusions

Comparaison gcn vs gcn+geo : le GCN agrège déjà les vecteurs via les arêtes —
les vecteurs osseux explicites peuvent être redondants (les coordonnées suffisent à les
calculer) ou au contraire simplifier l'apprentissage.
